In [5]:
import os
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm

# Cấu hình đường dẫn (Sửa lại theo nơi bạn lưu dataset gốc)
# Cấu trúc thư mục gốc mong đợi:
# data/
# ├── WIDER_train/images/...
# ├── WIDER_val/images/...
# └── wider_face_split/wider_face_train_bbx_gt.txt, ...

DATASET_ROOT = '../data' 
OUTPUT_DIR = '../data'

def convert_box(size, box):
    # box: x, y, w, h
    dw = 1. / size[1]
    dh = 1. / size[0]
    x = box[0] + box[2] / 2.0
    y = box[1] + box[3] / 2.0
    w = box[2]
    h = box[3]
    
    x = x * dw
    w = w * dw
    y = y * dh
    h = h * dh
    return (x, y, w, h)

def process_set(set_name, split_file, img_root):
    save_img_dir = Path(OUTPUT_DIR) / 'images' / set_name
    save_label_dir = Path(OUTPUT_DIR) / 'labels' / set_name
    save_img_dir.mkdir(parents=True, exist_ok=True)
    save_label_dir.mkdir(parents=True, exist_ok=True)

    with open(split_file, 'r') as f:
        lines = f.readlines()

    idx = 0
    # Dùng tqdm để theo dõi tiến độ
    pbar = tqdm(total=len(lines), desc=f"Processing {set_name}")
    
    while idx < len(lines):
        line = lines[idx].strip()
        
        # --- FIX ROBUSTNESS: Bỏ qua dòng trống hoặc dòng không phải tên ảnh ---
        if not line.endswith('.jpg'):
            idx += 1
            pbar.update(1)
            continue
            
        filename = line
        idx += 1
        
        # Đọc số lượng box
        try:
            num_boxes = int(lines[idx].strip())
        except ValueError:
            # Nếu vẫn lỗi, thử nhảy cóc để tìm lại đồng bộ (resync)
            idx += 1 
            pbar.update(1)
            continue
            
        idx += 1
        
        boxes = []
        if num_boxes == 0:
            # --- FIX QUAN TRỌNG ---
            # Nếu num_boxes = 0, WIDER FACE thường vẫn có 1 dòng dummy "0 0 0..."
            # Ta cần kiểm tra dòng tiếp theo, nếu không phải là tên file ảnh (.jpg) thì skip nó
            if idx < len(lines):
                next_line = lines[idx].strip()
                if not next_line.endswith('.jpg'):
                    idx += 1 # Skip dòng dummy
        else:
            for _ in range(num_boxes):
                if idx >= len(lines): break
                # x1, y1, w, h, ...
                val_strs = lines[idx].strip().split()
                # Chỉ lấy dòng có đủ dữ liệu
                if len(val_strs) >= 4:
                    boxes.append(list(map(float, val_strs[:4])))
                idx += 1

        # Cập nhật thanh tiến độ đúng số dòng đã đọc
        # (filename line + num_boxes line + box lines)
        pbar.update(2 + (1 if num_boxes == 0 else num_boxes))

        # --- COPY ẢNH VÀ GHI LABEL ---
        src_img_path = os.path.join(img_root, filename)
        
        # Kiểm tra ảnh tồn tại mới xử lý
        if not os.path.exists(src_img_path):
            continue
            
        img = cv2.imread(src_img_path)
        if img is None: continue
        h, w = img.shape[:2]

        dst_img_path = save_img_dir / Path(filename).name
        shutil.copy(src_img_path, dst_img_path)

        # Ghi file label (nếu có box)
        if len(boxes) > 0:
            label_file = save_label_dir / (Path(filename).stem + '.txt')
            with open(label_file, 'w') as lf:
                for box in boxes:
                    # box: x1, y1, w, h
                    # Convert sang format YOLO: x_center, y_center, w, h (normalized)
                    xywh = convert_box((h, w), box)
                    
                    # Giới hạn giá trị trong khoảng [0, 1] để tránh lỗi
                    xc, yc, bw, bh = xywh
                    xc = max(0, min(1, xc))
                    yc = max(0, min(1, yc))
                    bw = max(0, min(1, bw))
                    bh = max(0, min(1, bh))
                    
                    lf.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")
# Chạy chuyển đổi
if __name__ == "__main__":
    # Train set
    process_set('train', 
                os.path.join(DATASET_ROOT, 'wider_face_split/wider_face_train_bbx_gt.txt'),
                os.path.join(DATASET_ROOT, 'WIDER_train/images'))
    # Val set (Dùng làm Validation)
    process_set('val', 
                os.path.join(DATASET_ROOT, 'wider_face_split/wider_face_val_bbx_gt.txt'),
                os.path.join(DATASET_ROOT, 'WIDER_val/images'))
    # Test set (Dùng làm Test, vì WIDER FACE test set không có public labels)
    
    # Tạo file data.yaml cho YOLO
    yaml_content = f"""
path: {os.path.abspath(OUTPUT_DIR)}
train: images/train
val: images/val
test: images/val  # WIDER FACE test set không có public labels, ta dùng val để test
names:
  0: face
"""
    with open(f"{OUTPUT_DIR}/data.yaml", "w") as f:
        f.write(yaml_content)
    
    print("Dataset conversion complete.")

Processing val: 100%|██████████| 46160/46160 [02:05<00:00, 367.41it/s] 

Dataset conversion complete.


In [ ]:
!yolo detect train model=yolo26n.pt data=../data/data.yaml epochs=100 imgsz=640 batch=16 project=Smart_Door name=yolo26n_face verbose=True exist_ok=True